# Low-rank Predicate Text Decomposition Analysis

This notebook isolates the current predicate prompt + low-rank decomposition module from the scene-graph model. It answers one question: can the low-rank module itself reconstruct and organize CLIP predicate text features well enough to be a useful relation classifier basis?

It does not load the detector, relation head, image crops, or SGG training loop.

## What To Look For

- `recon_cos_mean`: should be high. If it stays low, the rank or initialization is not enough.
- `offdiag_abs_mean`: lower means reconstructed predicate classifiers are less collapsed.
- `top1_match`: how often each reconstructed feature is closest to its original predicate feature.
- `W_active_mean`: average number of active basis components per predicate; lower means sparser address/composition.
- `base_to_novel_cos_gap`: whether novel predicates remain separable from base predicates after reconstruction.

A good decomposition usually has high reconstruction cosine, high top-1 self match, non-trivial sparsity, and no obvious collapse to frequent predicates.

In [ ]:
# Parameters
from pathlib import Path
import os
import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'maskrcnn_benchmark').exists():
    PROJECT_ROOT = Path('/Users/shangfei/Developer/SDSGG')

PROMPT_JSON = os.environ.get(
    'PREDICATE_PROMPT_JSON',
    '/workspace/ccloud/sf/SDSGG/analysis/predicate_factor_descriptions.json',
)
PROMPT_FIELD = 'core_prompt'

RANKS = [4, 8, 16, 24, 32]
TRAIN_RANK = 16
STEPS = 1500
LR = 1e-3
TRAIN_BASIS = True
TRAIN_MODE = 'w'  # w: rel classifier path would update W only; decomposition losses still update B if TRAIN_BASIS=True
INIT_METHOD = 'pca'

RECON_LOSS_WEIGHT = 1.0
SPARSITY_WEIGHT = 0.02
BASIS_DECORR_WEIGHT = 5e-4
WEIGHT_DECORR_WEIGHT = 2e-4
SPARSITY_THRESHOLD = 0.05

DEVICE = os.environ.get('LOW_RANK_DEVICE', 'cuda' if torch.cuda.is_available() else 'cpu')
print('project:', PROJECT_ROOT)
print('prompt:', PROMPT_JSON)
print('device:', DEVICE)

In [ ]:
# Imports and repo modules
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import json
import math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import importlib

from CLIP import clip
from maskrcnn_benchmark.config import cfg
from maskrcnn_benchmark.modeling.roi_heads.relation_head.low_rank_text import (
    VG_PREDICATES,
    load_relation_prompt_texts,
    build_predicate_splits,
)
import maskrcnn_benchmark.modeling.roi_heads.relation_head.hola_low_rank as hola_low_rank
importlib.reload(hola_low_rank)
HOLaLowRankDecomposer = hola_low_rank.HOLaLowRankDecomposer

torch.set_grad_enabled(True)
torch.manual_seed(2026)
np.random.seed(2026)

# Make notebook tables readable: do not collapse long strings or hide columns with ellipses.
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 220)
pd.set_option('display.expand_frame_repr', False)

def explain(items):
    for name, meaning in items:
        print(f'- {name}: {meaning}')

In [ ]:
# Load current prompt text and encode with CLIP text encoder only
prompt_path = Path(PROMPT_JSON)
if not prompt_path.exists():
    candidates = [
        PROJECT_ROOT / 'analysis' / 'predicate_factor_descriptions.json',
        PROJECT_ROOT / 'predicate_factor_descriptions.json',
        Path('/workspace/ccloud/sf/SDSGG/analysis/predicate_factor_descriptions.json'),
    ]
    for candidate in candidates:
        if candidate.exists():
            prompt_path = candidate
            break
if not prompt_path.exists():
    raise FileNotFoundError(f'Prompt JSON not found. Set PROMPT_JSON or PREDICATE_PROMPT_JSON. Tried: {prompt_path}')

predicate_names = list(VG_PREDICATES)
fg_predicate_names = predicate_names[1:]
relation_texts = load_relation_prompt_texts(str(prompt_path), predicate_names, PROMPT_FIELD)
assert len(relation_texts) == len(fg_predicate_names)

clip_model, _ = clip.load('ViT-B/32', device=DEVICE)
clip_model.eval()
for p in clip_model.parameters():
    p.requires_grad = False

with torch.no_grad():
    tokens = clip.tokenize(relation_texts).to(DEVICE)
    text_features = clip_model.encode_text(tokens).float()
    text_features = F.normalize(text_features, dim=-1)

print('foreground predicates:', len(fg_predicate_names))
print('text feature shape:', tuple(text_features.shape))
print('Prompt preview:')
explain([
    ('predicate', 'foreground predicate 名称，不包含背景类。'),
    ('prompt', '当前送入 CLIP text encoder 的关系描述 prompt。'),
])
pd.DataFrame({'predicate': fg_predicate_names, 'prompt': relation_texts}).head(10)

In [ ]:
# Split metadata: indices here are foreground-only, so global predicate id k maps to fg index k - 1.
splits_global = build_predicate_splits(cfg)
splits_fg = {
    name: torch.as_tensor(ids[1:], device=DEVICE, dtype=torch.long) - 1
    for name, ids in splits_global.items()
}
for name, ids in splits_fg.items():
    print(name, len(ids), [fg_predicate_names[i] for i in ids[:8].detach().cpu().tolist()])

In [ ]:
# Metric helpers
def pairwise_offdiag_abs(x):
    x = F.normalize(x.float(), dim=-1)
    sim = x @ x.t()
    mask = ~torch.eye(sim.size(0), dtype=torch.bool, device=sim.device)
    return sim[mask].abs().mean().item()

def reconstruction_metrics(original, reconstructed, weights, basis, threshold=0.05):
    original = F.normalize(original.float(), dim=-1)
    reconstructed = F.normalize(reconstructed.float(), dim=-1)
    cos = F.cosine_similarity(original, reconstructed, dim=-1)
    sim = reconstructed @ original.t()
    top1 = sim.argmax(dim=1)
    top5 = sim.topk(k=min(5, sim.size(1)), dim=1).indices
    eye = torch.arange(sim.size(0), device=sim.device)
    abs_w = weights.float().abs()
    active = (abs_w > threshold).float().sum(dim=1)
    max_share = abs_w.max(dim=1).values / abs_w.sum(dim=1).clamp_min(1e-6)
    basis_sim = F.normalize(basis.float(), dim=-1) @ F.normalize(basis.float(), dim=-1).t()
    basis_mask = ~torch.eye(basis_sim.size(0), dtype=torch.bool, device=basis_sim.device)
    return {
        'recon_cos_mean': cos.mean().item(),
        'recon_cos_min': cos.min().item(),
        'recon_cos_std': cos.std().item(),
        'top1_match': (top1 == eye).float().mean().item(),
        'top5_match': (top5 == eye[:, None]).any(dim=1).float().mean().item(),
        'orig_offdiag_abs_mean': pairwise_offdiag_abs(original),
        'recon_offdiag_abs_mean': pairwise_offdiag_abs(reconstructed),
        'basis_offdiag_abs_mean': basis_sim[basis_mask].abs().mean().item(),
        'W_abs_mean': abs_w.mean().item(),
        'W_active_mean': active.mean().item(),
        'W_active_max': active.max().item(),
        'W_max_share_mean': max_share.mean().item(),
    }

def component_norm_metrics(model):
    residual = (model.class_weights @ model.basis_feat).float()
    mean = model.text_mean.float()
    residual_norm = residual.norm(dim=-1)
    mean_norm = mean.norm().item()
    original_norm = model.original_text_features.float().norm(dim=-1)
    return {
        'mean_norm': mean_norm,
        'residual_norm_mean': residual_norm.mean().item(),
        'residual_norm_min': residual_norm.min().item(),
        'residual_norm_max': residual_norm.max().item(),
        'residual_to_mean_ratio': (residual_norm.mean() / mean.norm().clamp_min(1e-6)).item(),
        'mean_to_original_norm_ratio': (mean.norm() / original_norm.mean().clamp_min(1e-6)).item(),
    }

def split_separation_metrics(features):
    features = F.normalize(features.float(), dim=-1)
    out = {}
    base = splits_fg['base']
    novel = splits_fg['novel']
    for split_name, ids in splits_fg.items():
        sim = features[ids] @ features[ids].t()
        mask = ~torch.eye(sim.size(0), dtype=torch.bool, device=sim.device)
        out[f'{split_name}_intra_abs'] = sim[mask].abs().mean().item() if mask.any() else 0.0
    base_novel = features[base] @ features[novel].t()
    out['base_to_novel_abs'] = base_novel.abs().mean().item()
    out['base_to_novel_max'] = base_novel.max(dim=0).values.mean().item()
    return out

def describe_nearest_neighbors(original, reconstructed, names, k=5):
    sim = F.normalize(reconstructed.float(), dim=-1) @ F.normalize(original.float(), dim=-1).t()
    rows = []
    for i, name in enumerate(names):
        vals, idxs = sim[i].topk(k)
        rows.append({
            'predicate': name,
            'self_cos': sim[i, i].item(),
            'top_neighbors': ', '.join(f'{names[j]}:{v:.3f}' for v, j in zip(vals.tolist(), idxs.tolist())),
            'top1_is_self': int(idxs[0].item() == i),
        })
    return pd.DataFrame(rows)

In [ ]:
# Original, undecomposed predicate text-space similarity baseline.
orig_offdiag_abs_mean = pairwise_offdiag_abs(text_features)
orig_sim = F.normalize(text_features.float(), dim=-1) @ F.normalize(text_features.float(), dim=-1).t()
orig_mask = ~torch.eye(orig_sim.size(0), dtype=torch.bool, device=orig_sim.device)
orig_offdiag_signed_mean = orig_sim[orig_mask].mean().item()
orig_offdiag_abs_max = orig_sim[orig_mask].abs().max().item()
print('Original CLIP text similarity metrics:')
explain([
    ('orig_offdiag_signed_mean', '原始 predicate 文本特征两两 cosine 的平均值，不取绝对值；越高说明 CLIP 文本空间本来就越挤。'),
    ('orig_offdiag_abs_mean', '原始 predicate 文本特征两两 cosine 绝对值平均；右图黑色虚线 baseline。'),
    ('orig_offdiag_abs_max', '最相似的一对不同 predicate 的 cosine 绝对值；越高说明存在非常难分的类别对。'),
])
display(pd.DataFrame([{
    'orig_offdiag_signed_mean': orig_offdiag_signed_mean,
    'orig_offdiag_abs_mean': orig_offdiag_abs_mean,
    'orig_offdiag_abs_max': orig_offdiag_abs_max,
}]).round(4))

text_mean = text_features.float().mean(0)
text_residual = text_features.float() - text_mean.unsqueeze(0)
print('\nMean / residual scale metrics:')
explain([
    ('text_mean_norm', '所有 predicate 共享的 CLIP 语义中心 mean 的范数。'),
    ('text_residual_norm_mean/min/max', '每个 predicate 去掉 mean 后 residual 的范数统计；它主要承载类别差异。'),
    ('text_residual_to_mean_ratio', '平均 residual 范数 / mean 范数；越小说明公共方向越主导，低 rank 也容易得到高 cosine。'),
    ('text_mean_to_feature_norm_ratio', 'mean 范数 / 原始 feature 平均范数；越接近 1，说明原始文本特征越受公共 mean 主导。'),
])
display(pd.DataFrame([{
    'text_mean_norm': text_mean.norm().item(),
    'text_residual_norm_mean': text_residual.norm(dim=-1).mean().item(),
    'text_residual_norm_min': text_residual.norm(dim=-1).min().item(),
    'text_residual_norm_max': text_residual.norm(dim=-1).max().item(),
    'text_residual_to_mean_ratio': (text_residual.norm(dim=-1).mean() / text_mean.norm().clamp_min(1e-6)).item(),
    'text_mean_to_feature_norm_ratio': (text_mean.norm() / text_features.float().norm(dim=-1).mean().clamp_min(1e-6)).item(),
}]).round(4))

In [ ]:
# Rank sweep with PCA initialization, before gradient training.
rank_rows = []
for rank in RANKS:
    model = HOLaLowRankDecomposer(
        text_features,
        rank=rank,
        init_method=INIT_METHOD,
        train_basis=TRAIN_BASIS,
        train_mode=TRAIN_MODE,
        recon_loss_weight=RECON_LOSS_WEIGHT,
        sparsity_weight=SPARSITY_WEIGHT,
        basis_decorr_weight=BASIS_DECORR_WEIGHT,
        weight_decorr_weight=WEIGHT_DECORR_WEIGHT,
    ).to(DEVICE)
    with torch.no_grad():
        rec = model.reconstruct()
        row = {'rank': rank, **reconstruction_metrics(text_features, rec, model.class_weights, model.basis_feat, SPARSITY_THRESHOLD)}
        row.update(component_norm_metrics(model))
        row.update(split_separation_metrics(rec))
        rank_rows.append(row)

rank_df = pd.DataFrame(rank_rows)
print('Rank sweep metrics:')
explain([
    ('rank', '低秩分解使用的 basis 数量。'),
    ('recon_cos_mean/min/std', '原始文本特征和 mean + W @ B 重建特征的 cosine 统计；mean 会把该值拉高，不能单独判断区分度。'),
    ('top1_match/top5_match', '重建后的第 i 类最近邻是否仍是原始第 i 类；这是类别身份是否保住的关键指标。'),
    ('orig_offdiag_abs_mean vs recon_offdiag_abs_mean', '原始/重建类别间平均相似度；重建值明显更高说明低秩让类别更挤。'),
    ('basis_offdiag_abs_mean', 'basis 之间的平均非对角相似度；PCA 初始化通常天然接近 0。'),
    ('W_active_mean/W_active_max', '每个 predicate 使用了多少个超过阈值的 basis；越低越像稀疏地址。'),
    ('mean_norm/residual_norm_mean/residual_to_mean_ratio', 'mean 和 W @ B residual 的范数比例，用来看高 cosine 是否主要由 mean 贡献。'),
])
display(rank_df.round(4))
rank_df.plot(x='rank', y=['recon_cos_mean', 'top1_match', 'basis_offdiag_abs_mean'], marker='o', figsize=(8, 4), grid=True)
plt.title('PCA initialization quality by rank')
plt.show()

In [ ]:
# Train only the decomposition module. No image model and no relation classifier loss.
decomposer = HOLaLowRankDecomposer(
    text_features,
    rank=TRAIN_RANK,
    init_method=INIT_METHOD,
    train_basis=TRAIN_BASIS,
    train_mode=TRAIN_MODE,
    recon_loss_weight=RECON_LOSS_WEIGHT,
    sparsity_weight=SPARSITY_WEIGHT,
    basis_decorr_weight=BASIS_DECORR_WEIGHT,
    weight_decorr_weight=WEIGHT_DECORR_WEIGHT,
).to(DEVICE)

trainable_params = [p for p in decomposer.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=0.0)
initial_W = decomposer.class_weights.detach().clone()
initial_B = decomposer.basis_feat.detach().clone()
history = []

print('decomposer module:', HOLaLowRankDecomposer.__module__)
print('init method:', INIT_METHOD)
print('trainable tensors:', sum(1 for _ in trainable_params))

def eval_decomposer(step):
    with torch.no_grad():
        rec = decomposer.reconstruct()
        metrics = reconstruction_metrics(text_features, rec, decomposer.class_weights, decomposer.basis_feat, SPARSITY_THRESHOLD)
        metrics.update(component_norm_metrics(decomposer))
        metrics.update(split_separation_metrics(rec))
        metrics['delta_W_norm'] = (decomposer.class_weights.detach() - initial_W).float().norm().item()
        metrics['delta_B_norm'] = (decomposer.basis_feat.detach() - initial_B).float().norm().item()
        metrics['step'] = step
        return metrics

def scalar_losses(losses):
    return {k: v.detach().float().item() for k, v in losses.items()}

row0 = eval_decomposer(0)
losses0 = decomposer.losses()
row0.update(scalar_losses(losses0))
row0['total_loss'] = sum(losses0.values()).detach().float().item()
row0['grad_norm'] = 0.0
history.append(row0)
for step in range(1, STEPS + 1):
    optimizer.zero_grad()
    losses = decomposer.losses()
    total = sum(losses.values())
    total.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, 5.0)
    optimizer.step()
    if step % 50 == 0 or step == 1:
        row = eval_decomposer(step)
        row.update(scalar_losses(losses))
        row['total_loss'] = total.detach().float().item()
        row['grad_norm'] = float(grad_norm)
        history.append(row)

hist_df = pd.DataFrame(history)
loss_cols = [c for c in ['total_loss', 'recon', 'sparse', 'weight_decorr', 'basis_decorr', 'grad_norm', 'delta_W_norm', 'delta_B_norm'] if c in hist_df]
print('Training diagnostics, first logged steps:')
explain([
    ('total_loss', '低秩模块所有辅助 loss 的总和。'),
    ('recon', '重建损失，约束 mean + W @ B 接近原始 CLIP text feature。'),
    ('sparse', 'W 的 L1 稀疏损失，鼓励每个 predicate 只使用少数 basis。'),
    ('weight_decorr', '约束不同 basis 的使用模式不要高度重复。'),
    ('basis_decorr', '约束 basis 之间正交/去相关。'),
    ('grad_norm', '当前梯度范数；如果长期为 0，说明训练没有真正更新。'),
    ('delta_W_norm/delta_B_norm', '当前 W/B 与初始化 W/B 的距离；用于确认参数是否真的在动。'),
])
display(hist_df[['step'] + loss_cols].head().round(6))
print('\nTraining diagnostics, final logged steps:')
display(hist_df.tail().round(4))

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 4, figsize=(21, 4))
hist_df.plot(x='step', y=['recon_cos_mean', 'recon_cos_min', 'top1_match'], ax=axes[0], grid=True)
axes[0].set_title('Reconstruction quality')
hist_df.plot(x='step', y=['W_active_mean', 'W_max_share_mean'], ax=axes[1], grid=True)
axes[1].set_title('Weight sparsity / concentration')
hist_df.plot(x='step', y=['basis_offdiag_abs_mean', 'recon_offdiag_abs_mean'], ax=axes[2], grid=True)
axes[2].axhline(orig_offdiag_abs_mean, color='black', linestyle='--', linewidth=1.2, label='orig_offdiag_abs_mean')
axes[2].legend()
axes[2].set_title('Decorrelation / collapse checks')
loss_plot_cols = [c for c in ['total_loss', 'recon', 'sparse', 'weight_decorr', 'basis_decorr', 'grad_norm'] if c in hist_df]
hist_df.plot(x='step', y=loss_plot_cols, ax=axes[3], grid=True)
axes[3].set_title('Loss / gradient diagnostics')
plt.tight_layout()
plt.show()

In [ ]:
# Visual diagnostics: feature geometry, predicate identity, and W addresses.
from sklearn.decomposition import PCA

with torch.no_grad():
    original = F.normalize(text_features.float(), dim=-1)
    reconstructed = F.normalize(decomposer.reconstruct().float(), dim=-1)
    sim_rec_orig = reconstructed @ original.t()
    self_cos = sim_rec_orig.diag().detach().cpu().numpy()
    sim_without_self = sim_rec_orig.clone()
    sim_without_self.fill_diagonal_(-1e9)
    nearest_other = sim_without_self.max(dim=1).values.detach().cpu().numpy()
    identity_margin = self_cos - nearest_other
    top1_is_self = (sim_rec_orig.argmax(dim=1) == torch.arange(sim_rec_orig.size(0), device=DEVICE)).detach().cpu().numpy()
    W_abs = decomposer.class_weights.detach().float().abs().cpu().numpy()
    W_active = (W_abs > SPARSITY_THRESHOLD).sum(axis=1)

all_features = torch.cat([original, reconstructed], dim=0).detach().cpu().numpy()
xy = PCA(n_components=2).fit_transform(all_features)
ori_xy = xy[:len(fg_predicate_names)]
rec_xy = xy[len(fg_predicate_names):]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].scatter(ori_xy[:, 0], ori_xy[:, 1], s=38, label='original', alpha=0.85)
axes[0, 0].scatter(rec_xy[:, 0], rec_xy[:, 1], s=38, marker='x', label='reconstructed', alpha=0.85)
for i in range(len(fg_predicate_names)):
    axes[0, 0].plot([ori_xy[i, 0], rec_xy[i, 0]], [ori_xy[i, 1], rec_xy[i, 1]], color='gray', alpha=0.25, linewidth=0.8)
for i in np.argsort(self_cos)[:8]:
    axes[0, 0].annotate(fg_predicate_names[i], rec_xy[i], fontsize=8)
axes[0, 0].set_title('Original vs reconstructed text features, PCA-2D')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

disp = rec_xy - ori_xy
q = axes[0, 1].quiver(ori_xy[:, 0], ori_xy[:, 1], disp[:, 0], disp[:, 1], self_cos, angles='xy', scale_units='xy', scale=1, cmap='viridis')
axes[0, 1].scatter(ori_xy[:, 0], ori_xy[:, 1], s=16, color='black', alpha=0.5)
fig.colorbar(q, ax=axes[0, 1], label='self cosine')
axes[0, 1].set_title('Reconstruction displacement arrows')
axes[0, 1].grid(True, alpha=0.3)

colors = np.where(top1_is_self, 'tab:blue', 'tab:red')
axes[1, 0].scatter(self_cos, identity_margin, c=colors, s=44, alpha=0.85)
axes[1, 0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1, 0].set_xlabel('self cosine: cos(recon_i, original_i)')
axes[1, 0].set_ylabel('identity margin: self - nearest other')
axes[1, 0].set_title('Predicate identity margin')
axes[1, 0].grid(True, alpha=0.3)
for i in np.argsort(identity_margin)[:8]:
    axes[1, 0].annotate(fg_predicate_names[i], (self_cos[i], identity_margin[i]), fontsize=8)

order = np.argsort(W_active)
im = axes[1, 1].imshow(W_abs[order], aspect='auto', interpolation='nearest', cmap='magma')
axes[1, 1].set_xlabel('basis index')
axes[1, 1].set_ylabel('predicate, sorted by active W count')
axes[1, 1].set_title('Absolute W address heatmap')
fig.colorbar(im, ax=axes[1, 1], label='abs(W)')

plt.tight_layout()
plt.show()

In [ ]:
# Similarity matrix view: original CLIP space vs reconstructed space.
with torch.no_grad():
    original = F.normalize(text_features.float(), dim=-1)
    reconstructed = F.normalize(decomposer.reconstruct().float(), dim=-1)
    orig_sim_mat = (original @ original.t()).detach().cpu().numpy()
    recon_sim_mat = (reconstructed @ reconstructed.t()).detach().cpu().numpy()
    diff_sim_mat = recon_sim_mat - orig_sim_mat

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
vmin = min(orig_sim_mat.min(), recon_sim_mat.min())
vmax = max(orig_sim_mat.max(), recon_sim_mat.max())
im0 = axes[0].imshow(orig_sim_mat, vmin=vmin, vmax=vmax, cmap='viridis')
axes[0].set_title('Original cosine matrix')
fig.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(recon_sim_mat, vmin=vmin, vmax=vmax, cmap='viridis')
axes[1].set_title('Reconstructed cosine matrix')
fig.colorbar(im1, ax=axes[1], fraction=0.046)
lim = np.abs(diff_sim_mat).max()
im2 = axes[2].imshow(diff_sim_mat, vmin=-lim, vmax=lim, cmap='coolwarm')
axes[2].set_title('Reconstructed - original')
fig.colorbar(im2, ax=axes[2], fraction=0.046)
for ax in axes:
    ax.set_xlabel('predicate index')
    ax.set_ylabel('predicate index')
plt.tight_layout()
plt.show()

In [ ]:
# Per-predicate reconstruction report
with torch.no_grad():
    reconstructed = decomposer.reconstruct()
    report = describe_nearest_neighbors(text_features, reconstructed, fg_predicate_names, k=5)

print('Per-predicate reconstruction report:')
explain([
    ('predicate', '当前关系类别名称。'),
    ('self_cos', '该类别重建特征和自己原始文本特征的 cosine。'),
    ('top_neighbors', '重建特征在原始文本特征空间里的前 k 个最近邻；用于发现被混淆到哪些类别。'),
    ('top1_is_self', '最近邻是否为自己；0 表示类别身份已经丢失。'),
])
print('\nWorst reconstructed predicates:')
display(report.sort_values('self_cos').head(12).round(4))
print('\nPotential collapsed predicates, where nearest original is not itself:')
display(report[report['top1_is_self'] == 0].sort_values('self_cos').round(4))

In [ ]:
# Inspect basis-address weights W. Sparse and interpretable addresses should show a few dominant basis components per predicate.
with torch.no_grad():
    W = decomposer.class_weights.detach().float().cpu().numpy()
    absW = np.abs(W)
    top_components = []
    for i, name in enumerate(fg_predicate_names):
        idx = absW[i].argsort()[::-1][:5]
        top_components.append({
            'predicate': name,
            'l1': absW[i].sum(),
            'active': int((absW[i] > SPARSITY_THRESHOLD).sum()),
            'top_basis': ', '.join(f'b{j}:{W[i, j]:.3f}' for j in idx),
        })
W_report = pd.DataFrame(top_components)
print('W address report:')
explain([
    ('predicate', '当前关系类别名称。'),
    ('l1', '该 predicate 的 W 地址向量 L1 范数；越大说明整体权重幅度越大。'),
    ('active', f'abs(W_ij) > {SPARSITY_THRESHOLD} 的 basis 数量；越低越稀疏。'),
    ('top_basis', '权重绝对值最大的前 5 个 basis 及其有符号权重；这里已关闭列宽截断。'),
])
print('\nMost sparse predicates:')
display(W_report.sort_values('active').head(10))
print('\nMost dense predicates:')
display(W_report.sort_values('active', ascending=False).head(10))

In [ ]:
# W concentration diagnostics: are all predicates relying on the same few basis dimensions?
with torch.no_grad():
    W = decomposer.class_weights.detach().float().cpu().numpy()
    absW = np.abs(W)
    active_mask = absW > SPARSITY_THRESHOLD
    basis_active_count = active_mask.sum(axis=0)
    basis_l1_mass = absW.sum(axis=0)
    basis_l2_mass = np.sqrt((W ** 2).sum(axis=0))
    mass_prob = basis_l1_mass / max(basis_l1_mass.sum(), 1e-12)
    entropy = -(mass_prob * np.log(mass_prob + 1e-12)).sum()
    effective_basis = np.exp(entropy)
    sorted_mass = np.sort(mass_prob)[::-1]
    top1_share = sorted_mass[:1].sum()
    top3_share = sorted_mass[:3].sum()
    top5_share = sorted_mass[:5].sum()
    sorted_asc = np.sort(mass_prob)
    gini = (2 * np.arange(1, len(sorted_asc) + 1) @ sorted_asc) / (len(sorted_asc) * sorted_asc.sum().clip(min=1e-12)) - (len(sorted_asc) + 1) / len(sorted_asc)

basis_usage_df = pd.DataFrame({
    'basis': [f'b{i}' for i in range(absW.shape[1])],
    'active_count': basis_active_count,
    'active_ratio': basis_active_count / absW.shape[0],
    'l1_mass': basis_l1_mass,
    'l1_mass_share': mass_prob,
    'l2_mass': basis_l2_mass,
}).sort_values('l1_mass_share', ascending=False)

print('W concentration summary:')
explain([
    ('active_count', f'有多少个 predicate 在该 basis 上满足 abs(W_ij) > {SPARSITY_THRESHOLD}。'),
    ('active_ratio', 'active_count / predicate 数量；越高说明这个 basis 被越多类别共用。'),
    ('l1_mass_share', '该 basis 占全部 abs(W) 质量的比例；越集中说明 W 越依赖少数 basis。'),
    ('effective_basis', '由 l1_mass_share 熵得到的有效 basis 数；越小表示越集中。'),
    ('topK_share', 'l1 质量最大的前 K 个 basis 占全部 W 质量的比例。'),
    ('gini', 'basis 使用不均衡程度；越接近 1 越集中，越接近 0 越均匀。'),
])
display(pd.DataFrame([{
    'rank': absW.shape[1],
    'effective_basis': effective_basis,
    'top1_share': top1_share,
    'top3_share': top3_share,
    'top5_share': top5_share,
    'gini': gini,
    'max_active_ratio': basis_usage_df['active_ratio'].max(),
}]).round(4))

print('\nMost used basis dimensions:')
display(basis_usage_df.head(min(10, len(basis_usage_df))).round(4))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
basis_usage_df.sort_values('basis').plot(x='basis', y='active_count', kind='bar', ax=axes[0], legend=False)
axes[0].set_title('Number of predicates activating each basis')
axes[0].set_ylabel('active predicate count')
axes[0].tick_params(axis='x', rotation=90)

basis_usage_df.sort_values('basis').plot(x='basis', y='l1_mass_share', kind='bar', ax=axes[1], legend=False)
axes[1].set_title('Share of total abs(W) mass per basis')
axes[1].set_ylabel('l1 mass share')
axes[1].tick_params(axis='x', rotation=90)

axes[2].plot(np.arange(1, len(sorted_mass) + 1), np.cumsum(sorted_mass), marker='o')
axes[2].set_ylim(0, 1.02)
axes[2].set_xlabel('top-k basis by l1 mass')
axes[2].set_ylabel('cumulative W mass share')
axes[2].set_title('Cumulative concentration curve')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Base / novel diagnostic. This helps test whether the compact basis keeps novel predicate text features usable.
with torch.no_grad():
    rec = F.normalize(decomposer.reconstruct().float(), dim=-1)
    ori = F.normalize(text_features.float(), dim=-1)
    rows = []
    for split_name, ids in splits_fg.items():
        cos = F.cosine_similarity(ori[ids], rec[ids], dim=-1)
        rows.append({
            'split': split_name,
            'count': len(ids),
            'recon_cos_mean': cos.mean().item(),
            'recon_cos_min': cos.min().item(),
            'recon_cos_std': cos.std().item() if len(ids) > 1 else 0.0,
        })
split_df = pd.DataFrame(rows)
print('Base / novel reconstruction diagnostics:')
explain([
    ('split', '当前 predicate 划分，例如 base/novel/semantic/total。'),
    ('count', '该划分包含的 foreground predicate 数量。'),
    ('recon_cos_mean/min/std', '该划分内原始文本特征和重建特征的 cosine 统计。'),
])
display(split_df.round(4))

base = splits_fg['base']
novel = splits_fg['novel']
sim_bn = rec[novel] @ rec[base].t()
nearest_base = sim_bn.argmax(dim=1).detach().cpu().tolist()
novel_rows = []
for local_i, base_j in enumerate(nearest_base):
    novel_idx = novel[local_i].item()
    base_idx = base[base_j].item()
    novel_rows.append({
        'novel_predicate': fg_predicate_names[novel_idx],
        'nearest_base': fg_predicate_names[base_idx],
        'cos': sim_bn[local_i, base_j].item(),
    })
print('\nNearest base predicate for each novel predicate:')
explain([
    ('novel_predicate', 'novel 关系类别。'),
    ('nearest_base', '在重建空间里与该 novel 最相似的 base 类别。'),
    ('cos', 'novel 与 nearest_base 的 cosine；太高说明 novel 可能贴到某个 base 上。'),
])
display(pd.DataFrame(novel_rows).sort_values('cos', ascending=False).round(4))

## How To Judge Effectiveness

Use these rough thresholds as a sanity check, not as paper claims:

- If `top1_match` is below `0.8`, the decomposed classifier no longer preserves predicate identity well.
- If `recon_cos_min` is very low, inspect the worst predicates; these are likely where relation training will collapse.
- If `W_active_mean` is close to `rank`, the module is not really sparse/address-like.
- If `basis_offdiag_abs_mean` remains high, the basis vectors are not disentangled.
- If novel predicates all map to the same nearest base predicate, the basis is compact but not useful for open-vocabulary transfer.

Recommended next experiment: run this notebook with `TRAIN_BASIS=False` and compare it with `TRAIN_BASIS=True`. If fixed PCA basis already has high reconstruction and better stability, use fixed basis for the first SGG training stage.